# Análise Financeira - ClearBank

Notebook para ler, validar e analisar as transações bancárias do arquivo CSV.


## Criando o arquivo de transações

In [ ]:
# Criar arquivo csv com os dados de teste
# Temos aqui registros validos, alguns invalidos e alguns suspeitos pra testar tudo

dados = """id,data,cliente_id,tipo,valor,descricao,categoria
1,2026-01-05,CLI001,credito,3500.00,Salario janeiro,salario
2,2026-01-12,CLI002,debito,180.50,Supermercado,compra
3,2026-01-20,CLI001,debito,abc,Erro no sistema,compra
4,2026-02-03,,debito,200.00,Sem cliente,transferencia
5,2026-02-14,CLI003,credito,15000.00,Transferencia grande,transferencia
6,2026-02-18,CLI002,debito,320.00,Conta de luz,conta
7,2026-03-01,CLI001,credito,3500.00,Salario marco,salario
8,2026-03-10,CLI003,debito,99.90,Streaming,assinatura
9,2026-03-15,CLI002,credito,2800.00,Freelance,salario
10,2026-03-22,CLI001,debito,450.00,Aluguel,compra
11,2026-04-01,CLI001,credito,3500.00,Salario abril,salario
12,2026-04-08,CLI002,debito,75.00,Farmacia,compra
13,2026-04-14,CLI003,credito,12500.00,Resgate investimento,transferencia
14,2026-04-18,CLI001,debito,890.00,Passagem,compra
15,2026-04-25,CLI002,credito,1200.00,Bonus,salario
16,2026-05-02,CLI003,debito,340.00,Restaurante,compra
17,2026-05-10,CLI001,credito,3500.00,Salario maio,salario
18,2026-05-15,CLI002,debito,60.00,Streaming,assinatura
19,,CLI001,debito,100.00,Data invalida,compra
20,2026-05-20,CLI003,tipo_errado,200.00,Tipo invalido,compra"""

with open('transacoes.csv', 'w', encoding='utf-8') as f:
    f.write(dados)

print("arquivo criado!")


## Importações

In [ ]:
import csv
import json
from datetime import datetime

# constante pra identificar transacoes suspeitas
LIMITE_SUSPEITO = 10000.00


## Funções

In [ ]:
def ler_transacoes(arquivo):
    transacoes = []
    try:
        with open(arquivo, newline='', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            for linha in reader:
                transacoes.append(dict(linha))
    except FileNotFoundError:
        print(f"arquivo {arquivo} nao encontrado")
    return transacoes


def validar_transacao(linha):
    # valida cada campo e retorna None se algo estiver errado

    # id precisa ser numero
    try:
        int(linha['id'])
    except (ValueError, KeyError):
        return None

    # cliente nao pode ser vazio
    if not linha.get('cliente_id', '').strip():
        return None

    # data precisa estar no formato certo
    try:
        data = datetime.strptime(linha['data'].strip(), '%Y-%m-%d')
    except (ValueError, KeyError):
        return None

    # tipo so pode ser credito ou debito
    tipo = linha.get('tipo', '').strip().lower()
    if tipo not in ['credito', 'debito']:
        return None

    # valor precisa ser numero maior que zero
    try:
        valor = float(linha['valor'])
        if valor <= 0:
            return None
    except (ValueError, KeyError):
        return None

    return {
        'id': int(linha['id']),
        'data': data,
        'cliente_id': linha['cliente_id'].strip(),
        'tipo': tipo,
        'valor': valor,
        'descricao': linha.get('descricao', ''),
        'categoria': linha.get('categoria', ''),
        'mes': data.strftime('%Y-%m')
    }


def gerar_relatorio(validas):
    resumo = {}

    for t in validas:
        mes = t['mes']

        if mes not in resumo:
            resumo[mes] = {
                'quantidade': 0,
                'total_credito': 0,
                'total_debito': 0,
                'saldo': 0,
                'media': 0,
                'maior_valor': 0,
                'menor_valor': float('inf'),
                'valores': []
            }

        resumo[mes]['quantidade'] += 1
        resumo[mes]['valores'].append(t['valor'])

        if t['tipo'] == 'credito':
            resumo[mes]['total_credito'] += t['valor']
        else:
            resumo[mes]['total_debito'] += t['valor']

    # calcula as metricas finais de cada mes
    for mes in resumo:
        vals = resumo[mes]['valores']
        resumo[mes]['saldo'] = round(resumo[mes]['total_credito'] - resumo[mes]['total_debito'], 2)
        resumo[mes]['media'] = round(sum(vals) / len(vals), 2)
        resumo[mes]['maior_valor'] = max(vals)
        resumo[mes]['menor_valor'] = min(vals)
        resumo[mes]['total_credito'] = round(resumo[mes]['total_credito'], 2)
        resumo[mes]['total_debito'] = round(resumo[mes]['total_debito'], 2)
        del resumo[mes]['valores']

    return dict(sorted(resumo.items()))


def salvar_json(relatorio, n_validas, n_invalidas):
    # precisa converter datetime pra string antes de salvar
    suspeitas_json = []
    for t in relatorio['suspeitas']:
        suspeitas_json.append({
            'id': t['id'],
            'cliente_id': t['cliente_id'],
            'data': t['data'].strftime('%Y-%m-%d'),
            'valor': t['valor'],
            'descricao': t['descricao']
        })

    payload = {
        'gerado_em': datetime.now().strftime('%Y-%m-%d'),
        'total_transacoes_validas': n_validas,
        'total_transacoes_invalidas': n_invalidas,
        'resumo_mensal': relatorio['resumo'],
        'transacoes_suspeitas': suspeitas_json
    }

    with open('relatorio.json', 'w', encoding='utf-8') as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

    print("relatorio.json salvo!")


def exibir_relatorio(relatorio, n_validas, n_invalidas, periodo):
    print("\n===== RESUMO DA LEITURA =====")
    print(f"Periodo analisado: {periodo[0]} ate {periodo[1]}")
    print(f"Total de linhas lidas: {n_validas + n_invalidas}")
    print(f"Linhas validas: {n_validas}")
    print(f"Linhas invalidas: {n_invalidas}")

    print("\n===== RELATORIO MENSAL =====")
    for mes, dados in relatorio['resumo'].items():
        print(f"\nMes: {mes}")
        print(f"  Transacoes: {dados['quantidade']}")
        print(f"  Total credito: R$ {dados['total_credito']:,.2f}")
        print(f"  Total debito:  R$ {dados['total_debito']:,.2f}")
        print(f"  Saldo:         R$ {dados['saldo']:,.2f}")
        print(f"  Media:         R$ {dados['media']:,.2f}")
        print(f"  Maior valor:   R$ {dados['maior_valor']:,.2f}")
        print(f"  Menor valor:   R$ {dados['menor_valor']:,.2f}")

    print("\n===== TRANSACOES SUSPEITAS =====")
    if relatorio['suspeitas']:
        for t in relatorio['suspeitas']:
            print(f"ID: {t['id']} | Cliente: {t['cliente_id']} | Data: {t['data'].strftime('%Y-%m-%d')} | Valor: R$ {t['valor']:,.2f}")
    else:
        print("Nenhuma transacao suspeita encontrada.")


print("funcoes definidas!")


## Executando a análise

In [ ]:
# lendo o arquivo
linhas = ler_transacoes('transacoes.csv')

# validando cada linha
validas = []
invalidas = 0

for linha in linhas:
    resultado = validar_transacao(linha)
    if resultado:
        validas.append(resultado)
    else:
        invalidas += 1

print(f"Total de linhas lidas: {len(linhas)}")
print(f"Linhas validas: {len(validas)}")
print(f"Linhas invalidas: {invalidas}")

# separando as suspeitas
suspeitas = [t for t in validas if t['valor'] > LIMITE_SUSPEITO]

# calculando periodo
datas = [t['data'] for t in validas]
periodo = (min(datas).strftime('%Y-%m-%d'), max(datas).strftime('%Y-%m-%d'))

# gerando o resumo mensal
resumo = gerar_relatorio(validas)

relatorio = {
    'resumo': resumo,
    'suspeitas': suspeitas
}

# salvando e exibindo
salvar_json(relatorio, len(validas), invalidas)
exibir_relatorio(relatorio, len(validas), invalidas, periodo)


## Análise com pandas

In [ ]:
import pandas as pd

# lendo com pandas pra comparar com o resultado acima
df = pd.read_csv('transacoes.csv', dtype=str)

# mesma validacao que fiz manualmente
df['valor_num'] = pd.to_numeric(df['valor'], errors='coerce')
df['data_dt'] = pd.to_datetime(df['data'], format='%Y-%m-%d', errors='coerce')

df_valido = df[
    df['id'].str.strip().str.isnumeric() &
    df['cliente_id'].notna() & (df['cliente_id'].str.strip() != '') &
    df['data_dt'].notna() &
    df['tipo'].str.strip().isin(['credito', 'debito']) &
    (df['valor_num'] > 0)
].copy()

df_valido['mes'] = df_valido['data_dt'].dt.strftime('%Y-%m')

# agrupando por mes
creditos = df_valido[df_valido['tipo'] == 'credito'].groupby('mes')['valor_num'].sum()
debitos  = df_valido[df_valido['tipo'] == 'debito'].groupby('mes')['valor_num'].sum()
qtd      = df_valido.groupby('mes')['valor_num'].count()

resultado_pandas = pd.DataFrame({'creditos': creditos, 'debitos': debitos, 'qtd': qtd}).fillna(0)
resultado_pandas['saldo'] = resultado_pandas['creditos'] - resultado_pandas['debitos']

print("Resultado com pandas:")
print(resultado_pandas.round(2))


## Gráfico com matplotlib

In [ ]:
import matplotlib.pyplot as plt

meses = list(resumo.keys())
saldos = [resumo[m]['saldo'] for m in meses]
creditos = [resumo[m]['total_credito'] for m in meses]
debitos = [resumo[m]['total_debito'] for m in meses]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# grafico de saldo por mes
cores = ['green' if s >= 0 else 'red' for s in saldos]
ax1.bar(meses, saldos, color=cores)
ax1.set_title('Saldo por mes')
ax1.set_xlabel('Mes')
ax1.set_ylabel('R$')
ax1.tick_params(axis='x', rotation=30)

# credito vs debito
x = range(len(meses))
ax2.bar([i - 0.2 for i in x], creditos, width=0.4, label='Credito', color='steelblue')
ax2.bar([i + 0.2 for i in x], debitos,  width=0.4, label='Debito',  color='salmon')
ax2.set_title('Credito vs Debito por mes')
ax2.set_xlabel('Mes')
ax2.set_ylabel('R$')
ax2.set_xticks(list(x))
ax2.set_xticklabels(meses, rotation=30)
ax2.legend()

plt.tight_layout()
plt.savefig('grafico.png', dpi=120)
plt.show()
print("grafico.png salvo!")
